<a href="https://colab.research.google.com/github/Andrew-lee0615/value-at-risk-analysis/blob/main/04_monte_carlo_var.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf


In [3]:

tickers = ["NVDA","INTC", "AMD", "MU", "MRVL", "WDC", "AVGO", "SPY"]
prices = yf.download(tickers, period="5y", auto_adjust=True)["Close"]
prices.tail()


[*********************100%***********************]  8 of 8 completed


Ticker,AMD,AVGO,INTC,MRVL,MU,NVDA,SPY,WDC
Date,,,,,,,,
2026-10-12,594.239990,354.730011,101.349998,265.859985,988.884338,226.320007,778.099976,386.940002
2026-10-13,605.530029,357.309998,102.809998,268.730011,953.040039,228.520004,778.010010,380.709991
2026-10-14,570.820007,346.250000,98.120003,245.279999,914.130005,221.300003,769.599976,364.100006
2026-10-15,562.229980,344.329987,95.370003,241.240005,931.239990,217.229996,771.789978,356.450012
2026-10-16,528.750000,328.750000,89.849998,227.830002,878.840027,214.130005,763.250000,332.000000


In [4]:
log_returns = np.log(prices / prices.shift(1)).dropna()
log_returns.head()

Ticker,AMD,AVGO,INTC,MRVL,MU,NVDA,SPY,WDC
Date,,,,,,,,
2021-10-19,-0.000859,0.013869,0.013494,0.010073,0.004598,0.003056,0.007685,0.004502
2021-10-20,0.000516,-0.001961,0.002894,-0.009775,0.010307,-0.008425,0.003920,0.013564
2021-10-21,0.024946,0.009301,0.011314,-0.001787,0.005405,0.026299,0.002605,0.006890
2021-10-22,0.004098,0.002448,-0.124188,-0.011397,-0.016600,0.001497,-0.001037,0.004392
2021-10-25,0.020977,0.012839,-0.001011,0.020302,0.018346,0.019176,0.005349,-0.003864


In [5]:
stocks = ["NVDA","INTC","AMD","MU","MRVL","WDC","AVGO"]
port = log_returns[stocks].mean(axis=1)
port.head()

,0
Date,
2021-10-19,0.006962
2021-10-20,0.001017
2021-10-21,0.011767
2021-10-22,-0.019964
2021-10-25,0.012395


In [6]:
port_var_95 = -port.quantile(0.05)
port_var_99 = -port.quantile(0.01)
print(port_var_95)
print(port_var_99)


0.04230155497479399
0.06934890896025886


In [7]:
mu = port.mean()
sigma = port.std()

param_var_95 = 1.645 * sigma - mu
param_var_99 = 2.326 * sigma - mu
print(param_var_95)
print(param_var_99)

0.04285079989843967
0.061167473804616776


In [9]:
np.random.seed(1)
sims = np.random.normal(mu, sigma, 10000)

mc_var_95 = -np.percentile(sims, 5)
mc_var_99 = -np.percentile(sims, 1)
print(mc_var_95)
print(mc_var_99)

0.04216740757047076
0.0601489545629775


In [10]:
df = 4
scale = sigma / np.sqrt(df / (df - 2))

np.random.seed(1)
sims_t = mu + scale * np.random.standard_t(df, 10000)

mc_t_var_95 = -np.percentile(sims_t, 5)
mc_t_var_99 = -np.percentile(sims_t, 1)
print(mc_t_var_95)
print(mc_t_var_99)

0.03799685317918835
0.06904359257235546


In [16]:
compare = pd.DataFrame({
    "Historical": [port_var_95, port_var_99],
    "Parametric": [param_var_95, param_var_99],
    "MC Normal": [mc_var_95, mc_var_99],
    "MC t (df=4)": [mc_t_var_95, mc_t_var_99]
}, index=["95% VaR", "99% VaR"])
compare

,Historical,Parametric,MC Normal,MC t (df=4)
95% VaR,0.042302,0.042851,0.042167,0.037997
99% VaR,0.069349,0.061167,0.060149,0.069044


## Key Finding
Monte Carlo with a normal distribution gives almost the same result as parametric VaR (99%: 6.01% vs 6.12%), and both underestimate the historical 99% VaR (6.93%). Switching to a fat-tailed Student-t distribution (df=4) brought the 99% VaR to 6.90%, almost identical to historical. However, the t-model's 95% VaR fell below historical (3.80% vs 4.23%). There is no single model that fits every confidence level, so model choice should depend on which part of the tail matters.